# Movement

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)  # Use camera index 0 for the default camera

if not cap.isOpened():
    print("Error: Couldn't open camera.")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc('X','V','I','D')
out = cv2.VideoWriter("output.avi", fourcc, 5.0, (frame_width, frame_height))

ret, frame1 = cap.read()  # 1st frame

if not ret:
    print("Error: Couldn't read the first frame.")
    exit()

single_detection_made = False

while cap.isOpened():
    ret, frame2 = cap.read()  # Read next frame
    if not ret:
        print("Error: Couldn't read frame.")
        break

    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh, None, iterations=3)
    contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        (x, y, w, h) = cv2.boundingRect(contour)
        if cv2.contourArea(contour) < 900:
            continue
        cv2.rectangle(frame2, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame2, "Status: {}".format('Movement'), (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        single_detection_made = True  # Set the flag indicating a detection is made
        break  # Break out of the loop once a single detection is made

    image = cv2.resize(frame2, (frame_width, frame_height))
    out.write(image)
    cv2.imshow("feed", frame2)

    frame1 = frame2

    if cv2.waitKey(40) == 27 or single_detection_made:
        # Break out of the loop if 'Esc' key is pressed or a single detection is made
        break

# Display the last frame with the detection result
cv2.imshow("Last Frame with Detection", frame2)
cv2.waitKey(0)

cv2.destroyAllWindows()
cap.release()
out.release()

# ssd


In [2]:
import cv2
import numpy as np

# Load the pre-trained MobileNet SSD model and its configuration
net = cv2.dnn.readNetFromCaffe('MobileNetSSD_deploy.prototxt', 'MobileNetSSD_deploy.caffemodel')

# List of class labels for MobileNet SSD
classes = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
           "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"]

# Open a video capture object
cap = cv2.VideoCapture(0)  # Use camera index 0 for the default camera

if not cap.isOpened():
    print("Error: Couldn't open camera.")
    exit()

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        print("Error: Couldn't read frame.")
        break

    # Resize the frame for processing
    resized_frame = cv2.resize(frame, (300, 300))
    
    # Preprocess the frame for the MobileNet SSD model
    blob = cv2.dnn.blobFromImage(resized_frame, 0.007843, (300, 300), 127.5)

    # Set the input to the pre-trained model
    net.setInput(blob)

    # Run forward pass to get detection
    detections = net.forward()

    frame_height, frame_width = frame.shape[:2]

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > 0.1:  # Confidence threshold
            class_id = int(detections[0, 0, i, 1])
            class_name = classes[class_id]

            # Draw bounding box and label on the frame
            box = detections[0, 0, i, 3:7] * np.array([frame_width, frame_height, frame_width, frame_height])
            (startX, startY, endX, endY) = box.astype("int")
            cv2.rectangle(frame, (startX, startY), (endX, endY), (0, 255, 0), 2)
            y = startY - 15 if startY - 15 > 15 else startY + 15
            cv2.putText(frame, class_name, (startX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow("Object Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
